# Self-Play Data Generation (Colab)

Generate self-play games for training data.
Saves JSON files to Drive (resume-safe).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. Model checkpoint (e.g., `pillar2p_ep6.pt`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model — EDIT THIS to match your model filename
MODEL_NAME = 'pillar2p_ep6.pt'

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/{MODEL_NAME} /content/alphatrain/data/model.pt
assert os.path.exists('/content/alphatrain/data/model.pt'), 'Model copy failed!'
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from alphatrain.evaluate import load_model
net, ms = load_model('/content/alphatrain/data/model.pt', 'cpu')
del net
print(f'Model OK, max_score={ms}')

In [ ]:
# === EDIT THESE FOR EACH INSTANCE ===
# Instance 1: 50000-50500
# Instance 2: 50500-51000
# Instance 3: 51000-51500
SEED_START = 50000
SEED_END = 50500
# ====================================

SIMS = 1600
WORKERS = 12
BATCH_SIZE = 32
MAX_TURNS = 5000
SAVE_DIR = f'{DRIVE}/selfplay_v5'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, Workers: {WORKERS}, Max turns: {MAX_TURNS}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/model.pt \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves 15 \
    --max-turns {MAX_TURNS}